# 🧩 Mini-Lab: Multi-Step Prompts

**Module 4: Prompt Engineering & Security** | **Duration: ~30 min** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** what prompt chaining is and why breaking complex tasks into sequential steps improves quality
2. **Implement** a multi-step prompt chain where each step's output feeds into the next step's input
3. **Compare** a single monolithic prompt vs. a chained approach on the same task

## Target Concepts

| Concept | Description |
|---------|-------------|
| Prompt Chaining | Breaking a complex task into a sequence of simpler prompts where each step's output becomes input for the next step |

## Setup

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()  # Uses OPENAI_API_KEY from .env
MODEL = "gpt-4o-mini"

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

def chat(prompt, system="You are a helpful assistant."):
    """Send a single prompt and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content

print("✅ Ready")

✅ Ready


## What Is Prompt Chaining?

**Prompt chaining** is a technique where you decompose a complex task into a sequence of simpler prompts. Each step produces an output that becomes the input for the next step.

```
User Input → [Step 1: Extract] → [Step 2: Analyze] → [Step 3: Format] → Final Output
```

**Why chain instead of doing it all in one prompt?**
- Each step is simpler, so the model makes fewer mistakes
- You can inspect and debug intermediate results
- You can use different instructions (or even models) per step
- Complex reasoning becomes more reliable when broken into stages

## Step 1 — The Monolithic Approach (Single Prompt)

Let's start by asking the model to do everything in one shot: take a product review, extract key points, analyze sentiment per point, and produce a structured summary.

In [5]:
review = """
I bought the ProMax Blender 3000 last month. The motor is incredibly powerful — it 
crushes ice in seconds and makes the smoothest smoothies I've ever had. The build 
quality feels premium with its stainless steel base. However, it's EXTREMELY loud, 
like jet-engine loud. My roommate complained from two rooms away. Also, the lid 
doesn't seal properly and I've had spinach smoothie splash onto my ceiling twice. 
Customer service was responsive when I reported the lid issue and they're sending 
a replacement. Overall, great blending power but the noise and lid design need work.
""".strip()

print(f"Review length: {len(review)} characters")

Review length: 582 characters


In [6]:
single_prompt = f"""Analyze this product review. Do ALL of the following:
1. Extract the key points mentioned
2. Classify each point as positive, negative, or neutral
3. Write a 2-sentence executive summary
4. Give an overall sentiment score from 1-10
5. List recommended improvements for the product team

Review:
{review}"""

single_result = chat(single_prompt)
md(single_result)

1. **Key Points Extracted:**
   - Powerful motor that crushes ice quickly
   - Makes the smoothest smoothies
   - Premium build quality with stainless steel base
   - Extremely loud operation
   - Lid does not seal properly, causing spills
   - Responsive customer service regarding the lid issue
   - Replacement lid being sent
   - Overall good blending power, but noise and lid design need improvement

2. **Classification of Each Point:**
   - Powerful motor that crushes ice quickly: Positive
   - Makes the smoothest smoothies: Positive
   - Premium build quality with stainless steel base: Positive
   - Extremely loud operation: Negative
   - Lid does not seal properly, causing spills: Negative
   - Responsive customer service regarding the lid issue: Positive
   - Replacement lid being sent: Positive
   - Overall good blending power, but noise and lid design need improvement: Neutral

3. **Executive Summary:**
   The ProMax Blender 3000 features a powerful motor that excels at making smoothies and has a premium build quality. However, it suffers from excessive noise and a poorly designed lid that causes spills, although customer service has been responsive to issues.

4. **Overall Sentiment Score:** 7/10

5. **Recommended Improvements for the Product Team:**
   - Reduce the noise level of the blender to make it more user-friendly.
   - Redesign the lid to ensure a proper seal and prevent spills during use.
   - Consider adding sound-dampening features or materials to enhance the user experience.

That works, but the model had to juggle five sub-tasks at once. Let's see how chaining compares.

## Step 2 — The Chained Approach (Multi-Step Prompts)

We'll break the same analysis into **three focused steps**, passing each output forward.

```
Review → [Step 1: Extract key points] 
             → [Step 2: Sentiment per point + score] 
                  → [Step 3: Executive summary + recommendations]
```

### Chain Step 1: Extract Key Points

In [7]:
step1_prompt = f"""Extract the distinct key points from this product review.
List each point as a single bullet. Only extract — do not analyze or judge.

Review:
{review}"""

step1_output = chat(step1_prompt)

md("### Step 1 Output: Key Points\n" + step1_output)

### Step 1 Output: Key Points
- The ProMax Blender 3000 has an incredibly powerful motor.
- It crushes ice in seconds.
- It makes the smoothest smoothies.
- The build quality feels premium with a stainless steel base.
- The blender is extremely loud, comparable to a jet engine.
- The lid does not seal properly.
- Spinach smoothies have splashed onto the ceiling due to the lid issue.
- Customer service is responsive and is sending a replacement for the lid.
- Overall, it has great blending power but the noise and lid design need improvement.

### Chain Step 2: Analyze Sentiment per Point

Now we feed Step 1's output into Step 2. The model only has to classify — it doesn't need to extract *and* classify at the same time.

In [8]:
step2_prompt = f"""For each key point below, classify the sentiment as Positive, Negative, or Neutral.
Then give an overall sentiment score from 1 (very negative) to 10 (very positive).

Key Points:
{step1_output}"""

step2_output = chat(step2_prompt)

md("### Step 2 Output: Sentiment Analysis\n" + step2_output)

### Step 2 Output: Sentiment Analysis
Here’s the sentiment classification for each key point:

1. The ProMax Blender 3000 has an incredibly powerful motor. - **Positive**
2. It crushes ice in seconds. - **Positive**
3. It makes the smoothest smoothies. - **Positive**
4. The build quality feels premium with a stainless steel base. - **Positive**
5. The blender is extremely loud, comparable to a jet engine. - **Negative**
6. The lid does not seal properly. - **Negative**
7. Spinach smoothies have splashed onto the ceiling due to the lid issue. - **Negative**
8. Customer service is responsive and is sending a replacement for the lid. - **Positive**
9. Overall, it has great blending power but the noise and lid design need improvement. - **Neutral**

Now, let's assign a sentiment score based on the overall sentiment:

- Positive points: 5 (4 points + 1 for customer service)
- Negative points: 3
- Neutral points: 1

To calculate the overall sentiment score, we can consider the positive points as adding to the score and negative points as subtracting from it. 

If we assign:
- Positive = +1
- Negative = -1
- Neutral = 0

The calculation would be:
- Total Score = (5 positive points) - (3 negative points) + (0 neutral points) = 5 - 3 = 2

Given that the overall impression is more positive than negative, but there are significant issues, I would rate the overall sentiment score as **6** out of 10. 

So, the overall sentiment score is **6**.

### Chain Step 3: Executive Summary & Recommendations

Finally, we combine all prior context to produce the summary. This step benefits from having clean, structured input from Steps 1 and 2.

In [9]:
step3_prompt = f"""Based on the key points and sentiment analysis below, produce:
1. A 2-sentence executive summary of the review
2. A bullet list of recommended improvements for the product team

Key Points:
{step1_output}

Sentiment Analysis:
{step2_output}"""

step3_output = chat(step3_prompt)

md("### Step 3 Output: Summary & Recommendations\n" + step3_output)

### Step 3 Output: Summary & Recommendations
### Executive Summary
The ProMax Blender 3000 boasts an incredibly powerful motor that excels at crushing ice and creating smooth smoothies, all while featuring a premium stainless steel build. However, it suffers from significant drawbacks, including excessive noise and a poorly sealing lid, which can lead to messy spills during use.

### Recommended Improvements for the Product Team
- **Reduce Noise Level**: Investigate design modifications or materials that can help minimize the operational noise of the blender.
- **Enhance Lid Design**: Redesign the lid to ensure a secure seal during blending to prevent spills and splashes.
- **Conduct Rigorous Testing**: Implement thorough testing protocols to identify and resolve potential issues with lid sealing before product release.
- **Provide Clear Usage Instructions**: Include detailed guidelines on how to properly secure the lid to avoid user errors that may lead to spills.
- **Consider Customer Feedback**: Regularly gather and analyze customer feedback to identify areas for continuous improvement in product design and functionality.

## Step 3 — Building a Reusable Chain Function

In practice, you'd wrap prompt chains into a function so each step is clearly defined and the data flows automatically.

In [10]:
def analyze_review_chain(review_text, verbose=True):
    """Three-step prompt chain for review analysis."""
    
    # Step 1: Extract
    key_points = chat(
        f"Extract the distinct key points from this product review. "
        f"List each as a single bullet. Only extract — do not analyze.\n\n"
        f"Review:\n{review_text}"
    )
    if verbose:
        md("**Step 1 — Key Points:**\n" + key_points)
    
    # Step 2: Analyze
    sentiment = chat(
        f"Classify each key point as Positive, Negative, or Neutral. "
        f"Give an overall score from 1-10.\n\n"
        f"Key Points:\n{key_points}"
    )
    if verbose:
        md("---\n**Step 2 — Sentiment:**\n" + sentiment)
    
    # Step 3: Summarize
    summary = chat(
        f"Write a 2-sentence executive summary and list recommended improvements.\n\n"
        f"Key Points:\n{key_points}\n\n"
        f"Sentiment:\n{sentiment}"
    )
    if verbose:
        md("---\n**Step 3 — Summary & Recommendations:**\n" + summary)
    
    return {"key_points": key_points, "sentiment": sentiment, "summary": summary}

In [11]:
# Try the chain on a different review
new_review = """
The CloudWalk running shoes are super lightweight and the cushioning is amazing for 
long runs. My knee pain has actually decreased since switching to these. The breathable 
mesh keeps my feet cool. On the downside, the sizing runs small — I had to go up a 
full size. The color options are also quite limited, only black and grey. The price is 
fair for the quality you get.
""".strip()

result = analyze_review_chain(new_review)

**Step 1 — Key Points:**
- The CloudWalk running shoes are super lightweight.
- The cushioning is amazing for long runs.
- Knee pain has decreased since switching to these shoes.
- The breathable mesh keeps feet cool.
- Sizing runs small; had to go up a full size.
- Limited color options (only black and grey).
- The price is fair for the quality.

---
**Step 2 — Sentiment:**
Here’s the classification of each key point along with an overall score:

1. **The CloudWalk running shoes are super lightweight.** - Positive
2. **The cushioning is amazing for long runs.** - Positive
3. **Knee pain has decreased since switching to these shoes.** - Positive
4. **The breathable mesh keeps feet cool.** - Positive
5. **Sizing runs small; had to go up a full size.** - Negative
6. **Limited color options (only black and grey).** - Negative
7. **The price is fair for the quality.** - Positive

**Overall Classification:**
- Positive: 5
- Negative: 2
- Neutral: 0

**Overall Score: 8/10**

The overall score reflects the strong positive aspects of the shoes, with only a couple of negative points that may affect some users.

---
**Step 3 — Summary & Recommendations:**
**Executive Summary:**
The CloudWalk running shoes are highly praised for their lightweight design, exceptional cushioning for long runs, and effective knee pain relief, making them a solid choice for runners. However, potential buyers should be aware of the small sizing and limited color options, which may impact their overall satisfaction.

**Recommended Improvements:**
1. **Adjust Sizing:** Offer half sizes or a wider range of sizes to accommodate different foot shapes and prevent sizing issues.
2. **Expand Color Options:** Introduce additional color choices to appeal to a broader audience and enhance customer personalization.
3. **Enhance Marketing:** Highlight the shoe's benefits more prominently in marketing materials to attract potential buyers, especially those with knee pain concerns.

## When to Use Prompt Chaining

| Scenario | Single Prompt | Chaining |
|----------|:---:|:---:|
| Simple, one-step task | ✅ | Overkill |
| Task with 2+ distinct subtasks | ⚠️ | ✅ |
| Need to inspect intermediate results | ❌ | ✅ |
| Different instructions per step | ❌ | ✅ |
| Minimizing API calls matters | ✅ | ⚠️ More calls |

**Trade-off:** Chaining uses more API calls (and thus more cost/latency), but each call is simpler and more reliable. For complex tasks, the quality improvement is usually worth it.

## 🎯 Summary

### Key Takeaways

1. **Prompt chaining** — breaks a complex task into a sequence of simpler prompts where each step's output feeds into the next step's input
2. **Decomposition improves quality** — focused single-task prompts produce more reliable results than one monolithic prompt trying to do everything
3. **Debuggability** — chaining lets you inspect intermediate outputs so you can identify and fix exactly where things go wrong
4. **Reusable patterns** — wrapping chains into functions creates reliable, repeatable multi-step workflows